# 04 Ensemble Methods

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)



In [ ]:
# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from sklearn.datasets import make_classification
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'figure.figsize': (10, 6), 'figure.dpi': 150})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, linewidth=120, precision=4)

# --- Utility Functions ---



---

### Table of Contents

1.  [**The Wisdom of the Crowd: Introduction to Ensembling**](#intro)
2.  [**Bagging: Reducing Variance**](#bagging)
3.  [**Boosting: Reducing Bias**](#boosting)
4.  [**Stacking: Combining Heterogeneous Models**](#stacking)
5.  [**Code Lab: Comparing Ensemble Techniques**](#code-lab)
6.  [**Summary**](#summary)

## The Lens: The Wisdom of Crowds

**What problem are we solving?**
No single model is best for every dataset. **Ensemble methods** combine multiple models to reduce variance (bagging), bias (boosting), or both (stacking). Random Forests, the workhorse ensemble, build many decorrelated trees and average their predictions.

**Why this method?**
Ensembles consistently outperform individual models in practice. Random Forests are robust, require minimal tuning, and provide built-in feature importance—making them ideal for exploratory economic analysis.

### Learning Objectives
* **Distinguish** bagging, boosting, and stacking as ensemble strategies.
* **Implement** Random Forests and interpret out-of-bag error and feature importance.
* **Build** stacked ensembles that combine heterogeneous base learners.
* **Evaluate** ensemble performance against single-model baselines.

### Prerequisites
* **ML Basics:** Decision trees, bias-variance tradeoff (Module 07 - Intro).
* **Python:** Scikit-learn API and cross-validation workflows.

<a id='intro'></a>
## 1. The Wisdom of the Crowd: Introduction to Ensembling

Ensemble methods are techniques that combine the predictions of multiple individual models (often called "weak learners") to produce a final prediction that is more accurate and robust than any of the individual models. The core idea is to leverage the diversity of the models to cancel out their individual errors.

There are three main paradigms in ensemble learning:

<a id='bagging'></a>
## 2. Bagging: Reducing Variance

**Bagging**, which stands for **Bootstrap Aggregating**, is a technique that primarily aims to **reduce the variance** of a model. It works by training multiple instances of the same base model on different random subsets of the training data (drawn with replacement, i.e., bootstrapping). The final prediction is then made by averaging the predictions of all the models (for regression) or by a majority vote (for classification).

> **Historical Context: Bagging**
> Bagging was introduced by Leo Breiman in 1996. It was one of the first and most influential ensemble methods, and it helped to popularize the use of decision trees in machine learning. The Random Forest algorithm, which is a modification of bagging, is one of the most widely used machine learning algorithms today.

The most famous example of a bagging algorithm is the **Random Forest**.

![Bagging Process](../images/07-Machine-Learning/bagging_process.png)

<a id='boosting'></a>
## 3. Boosting: Reducing Bias

**Boosting** is a sequential process where each new model is trained to correct the errors of its predecessors. Unlike bagging, where models are trained in parallel, boosting trains models sequentially. Each subsequent model places more emphasis on the data points that were misclassified by the previous models. This process allows the ensemble to focus on the most difficult cases, thereby **reducing the overall bias**.

Popular boosting algorithms include **AdaBoost** and **Gradient Boosting (e.g., XGBoost, LightGBM)**.

> **Historical Context: Boosting**
> The first boosting algorithm, AdaBoost (Adaptive Boosting), was proposed by Yoav Freund and Robert Schapire in 1996. Their work on AdaBoost earned them the prestigious Gödel Prize in 2003. Gradient Boosting, a more generalized version of boosting, was later developed by Jerome Friedman.

![Boosting Process](../images/07-Machine-Learning/boosting_process.png)

<a id='stacking'></a>
## 4. Stacking: Combining Heterogeneous Models

**Stacking** (or Stacked Generalization) takes a slightly different approach. Instead of using a simple function (like averaging or voting) to combine the predictions of the base models, it uses another model—a **meta-learner**—to learn the best way to combine them. Typically, the base models are diverse (e.g., a mix of SVMs, decision trees, and logistic regression), and the meta-learner is often a simple model like a logistic regression.

> **Historical Context: Stacking**
> Stacking (Stacked Generalization) was introduced by David Wolpert in 1992. It is a powerful technique for combining multiple models to achieve better predictive performance and is frequently used in data science competitions.

The training process involves splitting the data, training the base models on one part, and then training the meta-learner on the predictions made by the base models on the other part.

![Stacking Process](../images/07-Machine-Learning/stacking_process.png)

<a id='code-lab'></a>
## 5. Code Lab: Comparing Ensemble Techniques

Let's compare the performance of Bagging, Boosting, and Stacking on a sample classification problem.

### Comparing Ensemble Methods

In [ ]:

# Generate a synthetic dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --- 1. Bagging Classifier ---
# We use a Decision Tree as the base estimator for our Bagging classifier.
bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=50, random_state=42)
bagging.fit(X_train, y_train)
bagging_acc = accuracy_score(y_test, bagging.predict(X_test))

In [ ]:
display(Markdown(f"> **Note:** Bagging Accuracy: {bagging_acc:.4f}"))

In [ ]:

# --- 2. Boosting Classifier (AdaBoost) ---
# We use a shallow Decision Tree as the base estimator for our AdaBoost classifier.
boosting = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=42)
boosting.fit(X_train, y_train)
boosting_acc = accuracy_score(y_test, boosting.predict(X_test))

In [ ]:
display(Markdown(f"> **Note:** Boosting (AdaBoost) Accuracy: {boosting_acc:.4f}"))

In [ ]:

# --- 3. Stacking Classifier ---
# We use a Decision Tree and a Logistic Regression as our base learners.
base_learners = [
    ('dt', DecisionTreeClassifier()),
    ('lr', LogisticRegression())
]
# We use a Logistic Regression as our meta-learner.
stacking = StackingClassifier(estimators=base_learners, final_estimator=LogisticRegression())
stacking.fit(X_train, y_train)
stacking_acc = accuracy_score(y_test, stacking.predict(X_test))

In [ ]:
display(Markdown(f"> **Note:** Stacking Accuracy: {stacking_acc:.4f}"))

<a id='summary'></a>
## 6. Summary

Ensemble methods are a cornerstone of modern machine learning. They provide a powerful framework for building high-performance models by combining the strengths of multiple, simpler models.
- **Bagging** is an excellent choice for reducing the variance of complex models (like deep decision trees).
- **Boosting** is highly effective at building accurate models from simple, high-bias learners.
- **Stacking** offers the most flexibility, allowing you to combine fundamentally different types of models to capture a wider range of patterns in the data.